# 🚗 JalanCerdas AI — Model Training
## YOLOv8s Road Damage Detection (6 Class)

Notebook ini akan:
1. ✅ Setup environment (GPU, dependencies)
2. ✅ Download & prepare dataset
3. ✅ Train model YOLOv8s
4. ✅ Evaluate model
5. ✅ Export ke TFLite untuk mobile
6. ✅ Download model hasil training

---

## 📦 Step 1: Setup Environment

In [ ]:
#@title 1.1 Clone Repository & Install Dependencies { display-mode: "form" }
#@markdown Klik tombol play untuk setup environment

import os
import subprocess
import sys

# Clone repo
if not os.path.exists('jalancerdas-ai'):
    print("📦 Cloning repository...")
    !git clone https://github.com/Srjeff27/jalancerdas-ai.git
else:
    print("✅ Repository already cloned")

# Change to ai-model directory
%cd jalancerdas-ai/ai-model

# Install dependencies
print("\n📦 Installing dependencies...")
!pip install -q ultralytics>=8.1.0 pyyaml kagglehub tqdm

# Verify installation
print("\n✅ Verifying installation...")
import ultralytics
print(f"  Ultralytics: {ultralytics.__version__}")

import torch
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"  GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    print("\n🎉 GPU ready for training!")
else:
    print("\n⚠️ No GPU detected! Go to Runtime → Change runtime type → GPU")

## 📥 Step 2: Download & Prepare Dataset

In [ ]:
#@title 2.1 Download Dataset dari Kaggle { display-mode: "form" }
#@markdown Dataset: kerusakan-jalan (road damage images from Indonesia)

import kagglehub
from pathlib import Path
import shutil

# Download dataset
print("📥 Downloading dataset...")
kaggle_path = kagglehub.dataset_download("habibiahmadim09/kerusakan-jalan")
print(f"  Downloaded to: {kaggle_path}")

# Find the actual dataset directory
kaggle_dir = Path(kaggle_path)

# Check for nested structure
if (kaggle_dir / "kerusakan-jalan").exists():
    source_dir = kaggle_dir / "kerusakan-jalan"
elif (kaggle_dir / "train").exists():
    source_dir = kaggle_dir
else:
    # Find the first directory with train/valid/test
    for item in kaggle_dir.rglob("train"):
        if item.is_dir():
            source_dir = item.parent
            break

print(f"  Source directory: {source_dir}")
print(f"  Contents: {list(source_dir.iterdir())}")

In [ ]:
#@title 2.2 Convert VOC XML → YOLO Format { display-mode: "form" }
#@markdown Konversi anotasi dari XML (Pascal VOC) ke TXT (YOLO)

import xml.etree.ElementTree as ET
from pathlib import Path
import shutil

# Class mapping
CLASS_NAMES = [
    "retak_memanjang",
    "pengelupasan_lapisan_permukaan",
    "lubang",
    "retak_kulit_buaya",
    "retak_blok",
    "retak_pinggir",
]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def voc_to_yolo(xml_path):
    """Convert VOC XML to YOLO format."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    w = int(root.find("size/width").text)
    h = int(root.find("size/height").text)
    
    lines = []
    for obj in root.findall("object"):
        name = obj.find("name").text
        if name not in CLASS_TO_ID:
            continue
        
        cls_id = CLASS_TO_ID[name]
        bbox = obj.find("bndbox")
        xmin = float(bbox.find("xmin").text)
        ymin = float(bbox.find("ymin").text)
        xmax = float(bbox.find("xmax").text)
        ymax = float(bbox.find("ymax").text)
        
        # Convert to YOLO format
        cx = ((xmin + xmax) / 2) / w
        cy = ((ymin + ymax) / 2) / h
        bw = (xmax - xmin) / w
        bh = (ymax - ymin) / h
        
        lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    
    return lines

# Prepare dataset
print("\n🔄 Converting dataset (VOC → YOLO)...")

dataset_dir = Path("dataset")
if dataset_dir.exists():
    shutil.rmtree(dataset_dir)

split_map = {"train": "train", "valid": "val", "test": "test"}
total_images = 0
total_labels = 0

for src_split, dst_split in split_map.items():
    src_dir = source_dir / src_split
    dst_img = dataset_dir / dst_split / "images"
    dst_lbl = dataset_dir / dst_split / "labels"
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)
    
    xml_files = sorted(src_dir.glob("*.xml"))
    split_count = 0
    
    for xml_file in xml_files:
        # Find matching image
        img_path = None
        for ext in IMG_EXTS:
            candidate = src_dir / (xml_file.stem + ext)
            if candidate.exists():
                img_path = candidate
                break
        
        if img_path is None:
            continue
        
        # Convert XML → YOLO txt
        yolo_lines = voc_to_yolo(xml_file)
        
        # Copy image
        shutil.copy2(img_path, dst_img / img_path.name)
        
        # Write label
        lbl_path = dst_lbl / (xml_file.stem + ".txt")
        lbl_path.write_text("\n".join(yolo_lines))
        
        split_count += 1
        total_labels += len(yolo_lines)
    
    total_images += split_count
    print(f"  {dst_split}: {split_count} images")

print(f"\n✅ Dataset ready! Total: {total_images} images, {total_labels} labels")

In [ ]:
#@title 2.3 Analyze Class Distribution { display-mode: "form" }
#@markdown Cek distribusi class untuk memastikan data seimbang

from pathlib import Path
from collections import Counter

# Count instances per class
class_counts = Counter()
label_files = list(Path("dataset/train/labels").glob("*.txt"))

for lbl_file in label_files:
    with open(lbl_file) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                class_counts[int(parts[0])] += 1

total = sum(class_counts.values())

print("\n📊 Class Distribution (Training Set)")
print("=" * 60)

max_count = max(class_counts.values()) if class_counts else 1
for cls_id, count in sorted(class_counts.items()):
    bar_len = int(count / max_count * 25)
    bar = "█" * bar_len
    pct = count / total * 100 if total > 0 else 0
    print(f"  {CLASS_NAMES[cls_id]:<35} {count:>5} ({pct:>5.1f}%) {bar}")

print("=" * 60)
print(f"  Total: {total} instances")

# Check imbalance
if class_counts:
    min_c = min(class_counts.values())
    max_c = max(class_counts.values())
    ratio = max_c / min_c if min_c > 0 else float('inf')
    if ratio > 3:
        print(f"\n  ⚠️ Class imbalance detected (ratio: {ratio:.1f}x)")
    else:
        print(f"\n  ✅ Class distribution is acceptable (ratio: {ratio:.1f}x)")

## 🏋️ Step 3: Train Model

In [ ]:
#@title 3.1 Train YOLOv8s { display-mode: "form" }
#@markdown ### Training Configuration
#@markdown - **Model**: YOLOv8s (small) — best balance for mobile
#@markdown - **Epochs**: 150 (can adjust below)
#@markdown - **Image Size**: 640x640
#@markdown - **Batch Size**: 16
#@markdown
#@markdown Training will take ~2-3 hours on Colab GPU

#@markdown ---
#@markdown **Advanced Settings:**
EPOCHS = 150 #@param {type:"integer"}
BATCH_SIZE = 16 #@param {type:"integer"}
IMAGE_SIZE = 640 #@param {type:"integer"}
MODEL = "yolov8s.pt" #@param ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt"]

import time
from ultralytics import YOLO

print("\n" + "=" * 60)
print("  🏋️ JalanCerdas AI — Training")
print("=" * 60)
print(f"  Model:    {MODEL}")
print(f"  Epochs:   {EPOCHS}")
print(f"  Batch:    {BATCH_SIZE}")
print(f"  ImgSz:    {IMAGE_SIZE}")
print(f"  Device:   GPU (CUDA)")
print("=" * 60)

# Load model
model = YOLO(MODEL)

# Train
start_time = time.time()

results = model.train(
    data="configs/data.yaml",
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMAGE_SIZE,
    device=0,  # GPU 0
    patience=40,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    erasing=0.3,
    project="runs",
    name="train_colab",
    exist_ok=True,
    verbose=True,
)

elapsed = time.time() - start_time
hours, rem = divmod(elapsed, 3600)
minutes, seconds = divmod(rem, 60)

print("\n" + "=" * 60)
print(f"  ✅ Training complete in {int(hours)}h {int(minutes)}m {int(seconds)}s")
print(f"  Save dir: {results.save_dir}")
print("=" * 60)

## 📊 Step 4: Evaluate Model

In [ ]:
#@title 4.1 Run Evaluation { display-mode: "form" }

from ultralytics import YOLO

# Load best model
best_pt = "runs/train_colab/weights/best.pt"
model = YOLO(best_pt)

# Run validation
print("\n📊 Running evaluation on validation set...")
results = model.val(data="configs/data.yaml", device=0, split="val")

print("\n" + "=" * 60)
print("  📈 Evaluation Results")
print("=" * 60)
print(f"  mAP@50:      {results.box.map50:.4f}")
print(f"  mAP@50-95:   {results.box.map:.4f}")
print(f"  Precision:   {results.box.mp:.4f}")
print(f"  Recall:      {results.box.mr:.4f}")

f1 = 2 * (results.box.mp * results.box.mr) / (results.box.mp + results.box.mr + 1e-8)
print(f"  F1 Score:    {f1:.4f}")
print("=" * 60)

# Per-class results
print("\n  Per-class AP@50:")
for i, ap in enumerate(results.box.ap50):
    if i < len(CLASS_NAMES):
        print(f"    {CLASS_NAMES[i]:<35} {ap:.4f}")

In [ ]:
#@title 4.2 Visualize Training Results { display-mode: "form" }

from IPython.display import Image, display
from pathlib import Path

# Show training results plot
results_img = Path("runs/train_colab/results.png")
if results_img.exists():
    print("\n📈 Training Results:")
    display(Image(str(results_img), width=800))
else:
    print("⚠️ Results plot not found")

# Show confusion matrix
confusion_img = Path("runs/train_colab/confusion_matrix.png")
if confusion_img.exists():
    print("\n🎯 Confusion Matrix:")
    display(Image(str(confusion_img), width=600))

## 📦 Step 5: Export to TFLite

In [ ]:
#@title 5.1 Export Model to TFLite (FP16) { display-mode: "form" }
#@markdown Export model ke format TFLite untuk deployment di Android

from ultralytics import YOLO
import shutil

# Load best model
best_pt = "runs/train_colab/weights/best.pt"
model = YOLO(best_pt)

# Export to TFLite FP16
print("\n📦 Exporting to TFLite (FP16)...")
export_path = model.export(format="tflite", half=True, simplify=True)

from pathlib import Path
tflite_file = Path(export_path)
size_mb = tflite_file.stat().st_size / (1024 * 1024)

print(f"\n✅ Exported successfully!")
print(f"  Path: {tflite_file}")
print(f"  Size: {size_mb:.2f} MB")

# Copy to mobile assets directory
mobile_models = Path("../mobile/assets/models")
if mobile_models.exists():
    dest = mobile_models / "pothole_yolo.tflite"
    shutil.copy2(tflite_file, dest)
    print(f"  📁 Copied to: {dest}")

# Also export FP32 for comparison
print("\n📦 Also exporting FP32 version...")
model_fp32 = YOLO(best_pt)
export_fp32 = model_fp32.export(format="tflite", half=False, simplify=True)
fp32_file = Path(export_fp32)
fp32_size = fp32_file.stat().st_size / (1024 * 1024)
print(f"  FP32 Size: {fp32_size:.2f} MB")

## ⬇️ Step 6: Download Model

In [ ]:
#@title 6.1 Download Trained Model { display-mode: "form" }
#@markdown Download model yang sudah di-training

from google.colab import files
from pathlib import Path

print("\n📥 Preparing downloads...")

# Files to download
downloads = []

# Best PyTorch model
best_pt = Path("runs/train_colab/weights/best.pt")
if best_pt.exists():
    downloads.append(best_pt)
    print(f"  ✅ {best_pt.name} ({best_pt.stat().st_size / (1024*1024):.2f} MB)")

# TFLite model
tflite_files = list(Path("runs/train_colab").glob("*.tflite"))
for tflite in tflite_files:
    downloads.append(tflite)
    print(f"  ✅ {tflite.name} ({tflite.stat().st_size / (1024*1024):.2f} MB)")

# Mobile assets TFLite
mobile_tflite = Path("../mobile/assets/models/pothole_yolo.tflite")
if mobile_tflite.exists() and mobile_tflite not in downloads:
    downloads.append(mobile_tflite)
    print(f"  ✅ {mobile_tflite.name} ({mobile_tflite.stat().st_size / (1024*1024):.2f} MB)")

if downloads:
    print(f"\n📥 Downloading {len(downloads)} files...")
    for f in downloads:
        files.download(str(f))
    print("\n✅ Downloads complete!")
else:
    print("\n⚠️ No model files found. Please run training first.")

In [ ]:
#@title 6.2 Download Everything (ZIP) { display-mode: "form" }
#@markdown Download semua hasil training sebagai ZIP

import shutil
from google.colab import files
from pathlib import Path

# Create zip of all training results
print("\n📦 Creating ZIP archive...")
shutil.make_archive("jalancerdas-model", 'zip', "runs/train_colab")

# Get file size
zip_file = Path("jalancerdas-model.zip")
size_mb = zip_file.stat().st_size / (1024 * 1024)
print(f"  ZIP created: {zip_file.name} ({size_mb:.2f} MB)")

# Download
print("\n📥 Downloading...")
files.download(str(zip_file))
print("\n✅ Download complete!")

## 🎉 Selesai!

### Next Steps:
1. **Copy model** ke `mobile/assets/models/pothole_yolo.tflite`
2. **Build APK**: `flutter build apk --release`
3. **Test** di HP dengan dashcam

### Model Performance:
- YOLOv8s typically achieves ~70% mAP@50
- Inference time ~80ms on mobile GPU
- Model size ~11MB (FP16 TFLite)

### Tips:
- Jika mAP rendah, coba tambah epochs atau data
- Jika overfitting, kurangi epochs atau tambah augmentation
- Jika underfitting, coba model lebih besar (yolov8m.pt)